# Módulo 6. Explotación y visualización de datos  
## Práctico en PySpark: Visualización con datos del Censo

En este práctico vamos a trabajar con el conjunto de datos del **Censo 2023 de Uruguay**, utilizando **PySpark** para preparar, transformar y resumir la información, y bibliotecas de Python para generar visualizaciones.

### Objetivos del práctico
Al finalizar este notebook, se espera que puedas:

- explorar un dataset grande con PySpark;
- limpiar y transformar variables de interés;
- construir visualizaciones **básicas**, **intermedias** y **más complejas**;
- interpretar resultados a partir de tablas agregadas y gráficos;
- combinar el trabajo distribuido de PySpark con visualizaciones en Python.

### Idea general de trabajo
En Big Data, muchas veces el volumen de datos hace que no sea práctico graficar directamente todas las filas. Por eso, una estrategia muy común es:

1. **leer y limpiar con PySpark**;
2. **agregar o resumir la información**;
3. **convertir solo el resultado necesario a pandas**;
4. **graficar**.

Ese flujo será el que usaremos en este práctico.

## Estructura del práctico

Trabajaremos con tres niveles de dificultad:

### Parte A. Visualizaciones básicas

### Parte B. Visualizaciones intermedias

### Parte C. Visualizaciones más complejas

> **Importante:** no todo el trabajo de visualización ocurre “dentro” de PySpark.  
> PySpark se usa sobre todo para **preparar** y **resumir** los datos.  
> Luego, para graficar, convertiremos resultados pequeños a pandas.

Enlace de los *Microdatos Censo 2023 Anonimizados*: https://www.gub.uy/instituto-nacional-estadistica/politicas-y-gestion/microdatos-censo-2023-anonimizados


Conjunto que vamos a trabajar en esta clase: https://www5.ine.gub.uy/documents/CENSO%202023/Microdatos/personas_ext_26_02.rar


Cuestionario del censo: https://www.gub.uy/instituto-nacional-estadistica/sites/instituto-nacional-estadistica/files/2025-02/Cuestionario_censo2023%20%281%29.pdf


Diccionario de variables: https://www5.ine.gub.uy/documents/CENSO%202023/Microdatos/Diccionario%20de%20variables%202023.xlsx

Enlace para profundizar en visualizaciones: https://www.data-to-viz.com/

## 1. Importación de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, avg, desc, asc, round as spark_round,
    sum as spark_sum, countDistinct
)

## 2. Crear la sesión de Spark

Si estás trabajando en Google Colab, Jupyter local o VS Code, esta celda crea la sesión que usaremos en el práctico.

In [ ]:
spark = SparkSession.builder \
    .appName("Modulo6_Visualizacion_Censo") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark

## 3. Carga del dataset

Usaremos el mismo archivo del práctico anterior:

- `personas_ext_26_02.csv`

Si tu archivo está en otra carpeta, cambia la ruta en la celda siguiente.

In [ ]:
ruta = "personas_ext_26_02.csv"

sdf = spark.read.csv(ruta, header=True, inferSchema=True)

## 4. Exploración inicial

In [ ]:
print("Cantidad de filas:", sdf.count())
print("Cantidad de columnas:", len(sdf.columns))

In [ ]:
print("\nEsquema:")
sdf.printSchema()

In [ ]:
sdf.show(5, truncate=False)

## 5. Selección de variables de trabajo

En este práctico vamos a trabajar con estas variables del censo:

- **PERNA01**: edad;
- **NIVELEDU**: máximo nivel educativo alcanzado;
- **PERPH02**: variable de sexo al nacer;
- **DEPARTAMENTO**: variable geográfica.

In [ ]:
sdf_abreviada = sdf.select(
    col("PERNA01").alias("Edad"),
    col("NIVELEDU"),
    col("PERPH02").alias("Sexo"),
    col("DEPARTAMENTO")
    )

In [ ]:
sdf_abreviada.show(5, truncate=False)

## 6. Preparación de un DataFrame de análisis

Vamos a quedarnos con las variables principales y a limpiar algunos valores.

### Criterios iniciales
- eliminar nulos en edad y nivel educativo;
- quedarnos con edades razonables;
- excluir códigos educativos especiales si aparecen (por ejemplo: `12`, `8888`, `9898`), tal como hiciste en el práctico anterior.

In [ ]:
sdf_abreviada.printSchema()

In [ ]:
sdf_abreviada = sdf_abreviada.withColumn("Edad", col("Edad").cast("int"))
sdf_abreviada = sdf_abreviada.withColumn("NIVELEDU", col("NIVELEDU").cast("int"))
sdf_abreviada = sdf_abreviada.withColumn("Sexo", col("Sexo").cast("int"))
sdf_abreviada = sdf_abreviada.withColumn("DEPARTAMENTO", col("DEPARTAMENTO").cast("int"))

In [ ]:
print("Cantidad de filas:", sdf_abreviada.count())

Excluir los valores menores a 109


In [ ]:
sdf_abreviada = sdf_abreviada.filter(col("Edad") < 109)

Excluir los valores
|      12|
|    8888|
|    9898|

In [ ]:
sdf_abreviada = sdf_abreviada.filter(~col("NIVELEDU").isin([12, 8888, 9898]))

## 7. Crear variables derivadas para visualización

Muchas visualizaciones funcionan mejor cuando agrupamos o etiquetamos variables.

Crearemos:

- una variable **grupo_edad**;
- una versión etiquetada y simplificada de **NIVELEDU**.

In [ ]:
sdf_abreviada = sdf_abreviada.withColumn(
    "grupo_edad",
    when(col("Edad") < 15, "0-14")
    .when(col("Edad") < 25, "15-24")
    .when(col("Edad") < 35, "25-34")
    .when(col("Edad") < 45, "35-44")
    .when(col("Edad") < 55, "45-54")
    .when(col("Edad") < 65, "55-64")
    .otherwise("65+")
)

La siguiente tabla detalla la codificación para el máximo nivel educativo alcanzado (NIVELEDU) según la fuente de datos.

| Código (NIVELEDU) | Descripción (Máximo Nivel Educativo Alcanzado) |
| :---------------: | :--------------------------------------------- |
| 0                 | Menor de 4 años                                |
| 1                 | Preescolar                                     |
| 2                 | Primaria común o especial                      |
| 4                 | Educación media básica o Ciclo Básico          |
| 5                 | Educación media superior o Bachillerato        |
| 6                 | Capacitaciones o cursos de UTU                 |
| 7                 | Magisterio o profesorado                       |
| 8                 | Terciario no universitario                     |
| 9                 | Universidad o similar                          |
| 10                | Postgrado (Diploma/Maestría/Doctorado)         |
| 12                | Nunca asistió                                  |

In [ ]:
sdf_abreviada = sdf_abreviada.withColumn(
    "nivel_edu_label",
    when(col("NIVELEDU") == 0, "0 - Menor de 4 años")
    .when(col("NIVELEDU") == 1, "1 - Preescolar")
    .when(col("NIVELEDU") == 2, "2 - Primaria común o especial")
    .when(col("NIVELEDU") == 4, "4 - Educación media básica o Ciclo Básico ")
    .when(col("NIVELEDU") == 5, "5 - Educación media superior o Bachillerato ")
    .when(col("NIVELEDU") == 6, "6 - Capacitaciones o cursos de UTU ")
    .when(col("NIVELEDU") == 7, "7 - Magisterio o profesorado")
    .when(col("NIVELEDU") == 8, "8 - Terciario no universitario")
    .when(col("NIVELEDU") == 9, "9 - Universidad o similar")
    .when(col("NIVELEDU") == 10, "10 - Postgrado (Diploma/Maestría/Doctorado)")
    .when(col("NIVELEDU") == 11, "Nunca asistió ")
    .otherwise("Código no etiquetado")
)


In [ ]:
sdf_abreviada.show(5, truncate=False)

### Parte A. Visualizaciones básicas
Gráficos simples para entender distribuciones y frecuencias:
- barras;
- histogramas;
- tablas resumidas.

En esta sección trabajaremos con gráficos simples, útiles para describir una variable a la vez.

## 8. Frecuencia por nivel educativo

### Objetivo
Visualizar cuántas personas aparecen en cada categoría de nivel educativo.

### Idea metodológica
1. Agrupamos en PySpark.
2. Ordenamos.
3. Convertimos el resultado a pandas.
4. Graficamos.

In [ ]:
sdf_abreviada.printSchema()

In [ ]:
tabla_edu = (
    sdf_abreviada.groupBy("NIVELEDU")
      .count()
      .orderBy("NIVELEDU")
)

tabla_edu.show(truncate=False)

In [ ]:
pdf_edu = tabla_edu.toPandas()

plt.figure(figsize=(12, 5))
plt.bar(pdf_edu["NIVELEDU"], pdf_edu["count"])
plt.xticks(rotation=75, ha="right")
plt.title("Cantidad de personas por nivel educativo")
plt.xlabel("Nivel educativo")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()


## 9. Histograma de edades

El histograma permite observar la distribución de una variable numérica.

> Como el gráfico trabaja mejor con una sola columna numérica ya preparada, tomaremos únicamente la columna `Edad` y la pasaremos a pandas.

In [ ]:
pdf_edad = sdf_abreviada.select("Edad").toPandas()

plt.figure(figsize=(10, 5))
plt.hist(pdf_edad["Edad"], bins=20)
plt.title("Distribución de edades")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
sdf_abreviada.show(5, truncate=False)

## 10. Cantidad de personas por grupo de edad

Ahora construimos un gráfico de barras usando una variable derivada que resume mejor la edad.

In [ ]:
tabla_grupo_edad = (
    sdf_abreviada.groupBy("grupo_edad")
    .count()
    .toPandas()
    .sort_values("grupo_edad")
)

plt.figure(figsize=(9, 5))
plt.bar(tabla_grupo_edad["grupo_edad"], tabla_grupo_edad["count"])
plt.title("Cantidad de personas por grupo de edad")
plt.xlabel("Grupo de edad")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

# Parte B. Visualizaciones intermedias
Gráficos que comparan grupos o muestran relaciones:
- barras ordenadas;
- promedios por categoría;
- boxplots;
- proporciones por grupo.


En esta parte empezamos a comparar grupos entre sí o a mostrar relaciones entre variables.

## 11. Promedio de edad por nivel educativo

Este gráfico muestra una idea distinta: ya no contamos personas, sino que calculamos un **resumen estadístico** para cada categoría.

In [ ]:
tabla_promedio_edad = (
    sdf_abreviada.groupBy("nivel_edu_label")
      .agg(spark_round(avg("Edad"), 2).alias("edad_promedio"))
      .orderBy("edad_promedio")
)

tabla_promedio_edad.show(truncate=False)

In [ ]:
pdf_promedio_edad = tabla_promedio_edad.toPandas()

plt.figure(figsize=(12, 5))
plt.barh(pdf_promedio_edad["nivel_edu_label"], pdf_promedio_edad["edad_promedio"])
plt.title("Edad promedio por nivel educativo")
plt.xlabel("Edad promedio")
plt.ylabel("Nivel educativo")
plt.tight_layout()
plt.show()

In [ ]:
pdf_promedio_edad = tabla_promedio_edad.toPandas()

plt.figure(figsize=(12, 5))
plt.barh(pdf_promedio_edad["nivel_edu_label"], pdf_promedio_edad["edad_promedio"])

for i, valor in enumerate(pdf_promedio_edad["edad_promedio"]):
    plt.text(valor, i, round(valor, 1), va="center")

plt.title("Edad promedio por nivel educativo")
plt.xlabel("Edad promedio")
plt.ylabel("Nivel educativo")
plt.tight_layout()
plt.show()

In [ ]:
sdf_abreviada.describe().show()

## 12. Boxplot de edad por nivel educativo

El boxplot es útil para comparar la distribución de una variable numérica entre grupos.

Como demasiadas categorías pueden volver el gráfico difícil de leer, nos quedaremos con algunos niveles más frecuentes.

In [ ]:
top_niveles = (
    sdf_abreviada.groupBy("nivel_edu_label")
    .count()
    .orderBy("count", ascending=False)
    .limit(6)
    .toPandas()["nivel_edu_label"]
    .tolist()
)

pdf_box = sdf_abreviada.select("nivel_edu_label", "Edad").toPandas()
pdf_box = pdf_box[pdf_box["nivel_edu_label"].isin(top_niveles)]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, nivel in enumerate(top_niveles):
    axes[i].boxplot(pdf_box[pdf_box["nivel_edu_label"] == nivel]["Edad"])
    axes[i].set_title(nivel, fontsize=9)
    axes[i].set_ylabel("Edad")
    axes[i].set_xticks([])

plt.tight_layout()
plt.show()

## 13. Relación entre edad y nivel educativo 

Un scatterplot con todos los registros puede ser excesivo.  
Por eso tomaremos una **muestra aleatoria** para visualizar mejor la nube de puntos.

In [ ]:
muestra = sdf_abreviada.sample(withReplacement=False, fraction=0.02, seed=42).select("Edad", "NIVELEDU").toPandas()

muestra["NIVELEDU_jitter"] = muestra["NIVELEDU"] + np.random.uniform(-0.15, 0.15, size=len(muestra))

plt.figure(figsize=(9, 5))
plt.scatter(muestra["Edad"], muestra["NIVELEDU_jitter"], alpha=0.3)
plt.title("Relación entre edad y nivel educativo (muestra)")
plt.xlabel("Edad")
plt.ylabel("Nivel educativo")
plt.tight_layout()
plt.show()

## 14. Proporciones por sexo al nacer y nivel educativo

A continuación construiremos una tabla cruzada entre nivel educativo y sexo al nacer, y la representaremos con barras apiladas.

Este tipo de visualización permite comparar la composición relativa de cada categoría educativa.


In [ ]:
tabla_pivot = sdf_abreviada.groupBy("Sexo", "nivel_edu_label").count().toPandas()
tabla_pivot = tabla_pivot.pivot(index="nivel_edu_label", columns="Sexo", values="count").fillna(0)
tabla_pivot = tabla_pivot.div(tabla_pivot.sum(axis=1), axis=0)

tabla_pivot = tabla_pivot.rename(columns={1: "1: Varón", 2: "2: Mujer"})

tabla_pivot.plot(kind="bar", stacked=True, figsize=(25, 10))

plt.title("Proporción por sexo al nacer dentro de cada nivel educativo", fontsize=18)
plt.xlabel("Nivel educativo", fontsize=14)
plt.ylabel("Proporción", fontsize=14)
plt.xticks(rotation=75, ha="right", fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title="Sexo", fontsize=12, title_fontsize=13, loc="upper right", bbox_to_anchor=(1, -0.08))

plt.tight_layout()
plt.show()

# Parte C. Visualizaciones más complejas

Visualizaciones que requieren más preparación previa:
- pirámide poblacional o gráficos por múltiples dimensiones.


Estas visualizaciones requieren más preparación previa y suelen ser muy útiles para análisis exploratorio más rico.

## 15. Pirámide poblacional por sexo al nacer y grupo de edad

Esta es una visualización más avanzada y muy utilizada en datos censales.

- un lado del gráfico se representa con valores negativos;
- el otro con valores positivos;
- así se genera la “pirámide”.

In [ ]:
tabla_piramide = sdf_abreviada.groupBy("grupo_edad", "Sexo").count().toPandas()
tabla_piramide = tabla_piramide.pivot(index="grupo_edad", columns="Sexo", values="count").fillna(0)
tabla_piramide = tabla_piramide.rename(columns={1: "1: Varón", 2: "2: Mujer"})

sexo1 = tabla_piramide.columns[0]
sexo2 = tabla_piramide.columns[1]

plt.figure(figsize=(9, 6))
plt.barh(tabla_piramide.index, -tabla_piramide[sexo1], label=sexo1)
plt.barh(tabla_piramide.index, tabla_piramide[sexo2], label=sexo2)
plt.title("Pirámide poblacional por sexo al nacer y grupo de edad")
plt.xlabel("Cantidad de personas")
plt.ylabel("Grupo de edad")
plt.legend()
plt.tight_layout()
plt.show()

Nota: en este gráfico una de las categorías de sexo se representa con valores negativos únicamente para facilitar la visualización en forma de pirámide. Las cantidades reales siguen siendo positivas.

## Gráfico de barras agrupadas

In [ ]:
tabla_piramide.plot(kind="bar", figsize=(10, 6))
plt.title("Cantidad de personas por grupo de edad y sexo al nacer")
plt.xlabel("Grupo de edad")
plt.ylabel("Cantidad de personas")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 18. Edad promedio por departamento

En esta sección construiremos una comparación territorial simple usando la variable **DEPARTAMENTO**.

La idea es observar si existen diferencias en la edad promedio entre departamentos.


In [ ]:
tabla_geo = (
    sdf_abreviada.groupBy("DEPARTAMENTO")
    .agg(
        count("*").alias("personas"),
        avg("Edad").alias("edad_promedio")
    )
    .filter(col("personas") >= 100)
    .orderBy("edad_promedio", ascending=False)
)

pdf_geo = tabla_geo.limit(15).toPandas()

plt.figure(figsize=(10, 6))
plt.barh(pdf_geo["DEPARTAMENTO"], pdf_geo["edad_promedio"])
plt.gca().invert_yaxis()
plt.title("Edad promedio por departamento")
plt.xlabel("Edad promedio")
plt.ylabel("Departamento")
plt.tight_layout()
plt.show()

Código correlativo del 1 al 19 comenzando por Montevideo y continuando alfabéticamente

### Codificación de la variable DEPARTAMENTO

El código de departamento va del **1 al 19**.  
Según la anotación del censo, comienza en **Montevideo** y luego continúa **en orden alfabético**.

| Código | Departamento |
|--------|--------------|
| 1 | Montevideo |
| 2 | Artigas |
| 3 | Canelones |
| 4 | Cerro Largo |
| 5 | Colonia |
| 6 | Durazno |
| 7 | Flores |
| 8 | Florida |
| 9 | Lavalleja |
| 10 | Maldonado |
| 11 | Paysandú |
| 12 | Río Negro |
| 13 | Rivera |
| 14 | Rocha |
| 15 | Salto |
| 16 | San José |
| 17 | Soriano |
| 18 | Tacuarembó |
| 19 | Treinta y Tres |

In [ ]:
tabla_geo = (
    sdf_abreviada.groupBy("DEPARTAMENTO")
    .agg(
        count("*").alias("personas"),
        avg("Edad").alias("edad_promedio")
    )
    .filter(col("personas") >= 100)
    .orderBy("edad_promedio", ascending=False)
)

pdf_geo = tabla_geo.limit(15).toPandas()

departamentos = {
    1: "Montevideo",
    2: "Artigas",
    3: "Canelones",
    4: "Cerro Largo",
    5: "Colonia",
    6: "Durazno",
    7: "Flores",
    8: "Florida",
    9: "Lavalleja",
    10: "Maldonado",
    11: "Paysandú",
    12: "Río Negro",
    13: "Rivera",
    14: "Rocha",
    15: "Salto",
    16: "San José",
    17: "Soriano",
    18: "Tacuarembó",
    19: "Treinta y Tres"
}

pdf_geo["DEPARTAMENTO"] = pdf_geo["DEPARTAMENTO"].map(departamentos)

plt.figure(figsize=(10, 6))
plt.barh(pdf_geo["DEPARTAMENTO"], pdf_geo["edad_promedio"])
plt.gca().invert_yaxis()
plt.title("Edad promedio por departamento")
plt.xlabel("Edad promedio")
plt.ylabel("Departamento")
plt.tight_layout()
plt.show()

### Nivel educativo promedio por departamento

A continuación, vamos a agrupar los datos por departamento y calcular el promedio de la variable **NIVELEDU**.

Este resultado nos permite explorar si, en promedio, algunos departamentos presentan valores más altos o más bajos en esta variable.

> Nota: como **NIVELEDU** es una variable categórica codificada con números, este promedio debe interpretarse solo como una referencia general.

In [ ]:
tabla_edu_depto = (
    sdf_abreviada.groupBy("DEPARTAMENTO")
    .agg(avg("NIVELEDU").alias("nivel_edu_promedio"))
    .orderBy("nivel_edu_promedio", ascending=False)
)

pdf_edu_depto = tabla_edu_depto.toPandas()

departamentos = {
    1: "Montevideo",
    2: "Artigas",
    3: "Canelones",
    4: "Cerro Largo",
    5: "Colonia",
    6: "Durazno",
    7: "Flores",
    8: "Florida",
    9: "Lavalleja",
    10: "Maldonado",
    11: "Paysandú",
    12: "Río Negro",
    13: "Rivera",
    14: "Rocha",
    15: "Salto",
    16: "San José",
    17: "Soriano",
    18: "Tacuarembó",
    19: "Treinta y Tres"
}

pdf_edu_depto["DEPARTAMENTO"] = pdf_edu_depto["DEPARTAMENTO"].map(departamentos)

plt.figure(figsize=(10, 6))
plt.barh(pdf_edu_depto["DEPARTAMENTO"], pdf_edu_depto["nivel_edu_promedio"])
plt.gca().invert_yaxis()
plt.title("Nivel educativo promedio por departamento")
plt.xlabel("Nivel educativo promedio")
plt.ylabel("Departamento")
plt.tight_layout()
plt.show()

En este práctico trabajaste visualización de datos con una lógica muy utilizada en análisis de Big Data:

- **PySpark** para leer, limpiar, filtrar y agregar;
- **pandas + matplotlib** para visualizar resultados ya resumidos.

Este enfoque permite trabajar con datasets grandes sin perder claridad al momento de comunicar hallazgos.

> En análisis de datos, visualizar no es solo “hacer gráficos”:  
> implica decidir **qué resumir**, **qué comparar** y **cómo comunicarlo** de forma comprensible.

## 19. Apagar la sesión de Spark

Ejecuta esta celda al final del trabajo.

In [ ]:
spark.stop()